In [ ]:
!pip install pandas openpyxl xlsxwriter

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import matplotlib.font_manager as fm # Import font_manager
import datetime # Import datetime for timestamps
from matplotlib.colors import LinearSegmentedColormap # Import for custom colormap

# =========================
# 0. Matplotlib 字体设置 (解决中文显示问题)
# =========================

# Clear matplotlib font cache aggressively
print("Clearing matplotlib font cache...")
!rm -rf ~/.cache/matplotlib
print("Matplotlib font cache cleared.")

# Check if wqy-microhei is installed, if not, install it
# A more direct check for the font file (Colab often has /usr/share/fonts/truetype/wqy/wqy-microhei.ttc)
wqy_font_path = Path("/usr/share/fonts/truetype/wqy/wqy-microhei.ttc")
if not wqy_font_path.exists():
    print("wqy-microhei font not found, attempting to install...")
    !sudo apt-get update
    !sudo apt-get install -y fonts-wqy-microhei
    print("wqy-microhei font installation attempted.")
else:
    print("wqy-microhei font already installed.")

# Rebuild font cache explicitly after installation/check
print("Rebuilding matplotlib font cache...")
fm._load_fontmanager(try_read_cache=False) # Reload the font manager to pick up new fonts. forces a full rescan.
print("Matplotlib font cache rebuilt.")

# Explicitly add the font to the font manager if it exists and wasn't found by default
if wqy_font_path.exists():
    try:
        # Check if the font name 'WenQuanYi Micro Hei' or 'wqy-microhei' is already known
        if not any(f.name == 'WenQuanYi Micro Hei' for f in fm.fontManager.ttflist) and \
           not any(f.name == 'wqy-microhei' for f in fm.fontManager.ttflist):
            fm.fontManager.addfont(str(wqy_font_path))
            print(f"'{wqy_font_path.name}' explicitly added to font manager.")
    except Exception as e:
        print(f"Error explicitly adding font {wqy_font_path.name}: {e}")

# Set font parameters
# 'WenQuanYi Micro Hei' is often the actual name for wqy-microhei.ttc
plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei', 'wqy-microhei', 'SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# Verify if the font is found by its internal name
if any(f.name == 'WenQuanYi Micro Hei' for f in fm.fontManager.ttflist):
    print("WenQuanYi Micro Hei font successfully found by matplotlib.")
elif any(f.name == 'wqy-microhei' for f in fm.fontManager.ttflist):
    print("wqy-microhei font successfully found by matplotlib.")
else:
    print("Warning: Neither WenQuanYi Micro Hei nor wqy-microhei font found by matplotlib. Encoding issues may persist.")


# =========================
# 1. 参数设置
# =========================

# Changed input file path and type to user-specified Excel file
input_path = "/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv生成数据2/attraction_kmeans_k6_distribution_user_file_processed_ratio_wide.xlsx"
output_dir = Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv生成数据2/cluster可视化")
output_dir.mkdir(parents=True, exist_ok=True)

# 如果您的公园名称列不是 park/attraction/name，请在这里手动指定
park_col = None   # 例如: "attraction_name"
# cluster_id_col is no longer needed for wide-format aggregated data

# Get current timestamp for filenames
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# =========================
# 2. 读取数据
# =========================

# Changed to read_excel for the new input file (was read_csv causing error as the input path is .xlsx)
df = pd.read_excel(input_path)

print("原始数据形状:", df.shape)
print("列名:", df.columns.tolist())

# =========================
# 3. 自动识别公园名称列
# =========================

if park_col is None:
    possible_cols = [
        "park", "park_name", "attraction", "attraction_name",
        "name", "景点", "景点名称", "公园", "公园名称"
    ]
    park_candidates = [c for c in df.columns if c in possible_cols]

    if len(park_candidates) > 0:
        park_col = park_candidates[0]
        print(f"自动识别到的公园名称列: '{park_col}'")
    else:
        # Fallback for wide Excel format: use the first non-numeric column as park_col
        non_numeric_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        if non_numeric_cols:
            park_col = non_numeric_cols[0]
            print(f"未找到常见公园名称列，使用第一个非数值列 '{park_col}' 作为公园名称列。")
        else:
            raise ValueError("无法找到合适的公园名称列。请检查 Excel 文件内容或手动设置 `park_col`。")
else:
    print(f"公园名称列已手动指定为: '{park_col}'")

print("使用的公园名称列:", park_col)

# =========================
# 4. 根据用户提供的字典替换景点名字
# =========================
print("\n尝试使用用户提供的字典进行景点名称替换...")
user_park_code_mapping = {

    "香山-植物园": "HG1",
    "圆明园": "HG2",
    "颐和园": "HG3",
    "八大处公园": "P1",
    "巴沟山水园": "P2",
    "百望山森林公园": "P3",
    "北京西山国家森林公园": "P4",
    "北坞公园": "P5",
    "畅春新园体育休闲广场": "P6",
    "茶棚公园": "P7",
    "国家植物园": "P8",
    "船营公园": "P9",
    "丹青圃公园": "P10",
    "东升八家郊野公园": "P11",
    "功德寺公园": "P12",
    "海淀公园": "P13",
    "荷清园": "P14",
    "六郎庄公园": "P15",
    "妙云寺公园": "P16",
    "南旱河公园": "P17",
    "上地公园": "P18",
    "石渠公园": "P19",
    "树村公园": "P20",
    "王庄公园": "P21",
    "万柳公园": "P22",
    "肖家河公园": "P23",
    "西水磨公园": "P24",
    "一亩园公园": "P25",
    "影湖楼公园": "P26",
    "玉东郊野公园": "P27",
    "中坞公园": "P28",

}

if park_col in df.columns:
    df[park_col] = df[park_col].apply(lambda x: user_park_code_mapping.get(x, x))
    print("部分景点名称已根据用户字典映射成功。")
else:
    print(f"警告: 未找到列 '{park_col}' 进行用户字典名称替换。")


# =========================
# 5. 计算每个景点各 cluster 的出现频率 (Logic for Excel wide format)
# =========================

# Ensure park_col exists
if park_col not in df.columns:
    raise ValueError(f"错误: 景点列 '{park_col}' 未在Excel文件中找到。")

# Set the park_col as index for easier manipulation
df_indexed = df.set_index(park_col)

# Identify cluster columns (all columns that are numeric and not the park_col)
cluster_count_cols = [c for c in df_indexed.columns if pd.api.types.is_numeric_dtype(df_indexed[c])]

if not cluster_count_cols:
    raise ValueError("无法找到包含集群计数的列。请检查Excel文件内容，确保集群计数为数值列。")

# The df_indexed with only cluster_count_cols are our cluster_counts
cluster_counts = df_indexed[cluster_count_cols].copy()
print("\n每个景点各 cluster 的原始计数 (来自Excel):")
display(cluster_counts.head())

# Calculate proportions
# sum(axis=1) for total images in each park
cluster_proportions = cluster_counts.div(cluster_counts.sum(axis=1), axis=0)

# Convert cluster column names to string if they are integers, for consistency with original code
cluster_proportions.columns = cluster_proportions.columns.astype(str)
cluster_cols = cluster_proportions.columns.tolist()

print("\n每个景点各 cluster 的出现频率 (比例):")
display(cluster_proportions.head())

# Define the custom order for parks
custom_park_order = [
    "HG1", "HG2", "HG3", "P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "P9",
    "P10", "P11", "P12", "P13", "P14", "P15", "P16", "P17", "P18", "P19", "P20",
    "P21", "P22", "P23", "P24", "P25", "P26", "P27", "P28"
]

# Reindex cluster_proportions DataFrame based on the custom order
# Use .reindex() with fill_value=0 to handle cases where a park might be missing in data
cluster_proportions = cluster_proportions.reindex(custom_park_order, fill_value=0)

# Remove any parks from the custom_park_order that are not present in the original data
# This step ensures that only existing parks are displayed, in the specified order.
existing_parks_in_order = [p for p in custom_park_order if p in cluster_proportions.index]
cluster_proportions = cluster_proportions.loc[existing_parks_in_order]

print("\n按指定顺序排序后的每个景点各 cluster 的出现频率 (比例):")
display(cluster_proportions.head())


# =========================
# 6. Prepare data for heatmap and stacked bar plot
# =========================

hotmap_data = cluster_proportions.copy()
bar_data = cluster_proportions.copy()

# =========================
# 7. 热力图 (Reuse existing code with new data)
# =========================

plot_data = hotmap_data.copy()

# Set values less than 0.01 to NaN to make them white on the heatmap
# For display, values >= 0.01 will be colored, values < 0.01 will be white.
plot_data_for_coloring = hotmap_data.copy()
plot_data_for_coloring[plot_data_for_coloring < 0.01] = np.nan

# Create a custom colormap: start with white for 0, then transition to YlGnBu
original_cmap = plt.cm.get_cmap('YlGnBu')
colors = original_cmap(np.linspace(0, 1, 256))
colors[0] = [1, 1, 1, 1] # Set the lowest color (for vmin=0) to white
custom_cmap = LinearSegmentedColormap.from_list("custom_YlGnBu", colors, N=256)

plt.figure(figsize=(12, max(8, len(plot_data.index) * 0.35)))

im = plt.imshow(
    plot_data_for_coloring.values, # Use the modified data for coloring
    aspect="auto",
    interpolation="nearest",
    cmap=custom_cmap, # Use the custom colormap
    vmin=0,
    vmax=0.4
)

plt.colorbar(im, label="Cluster proportion")

plt.xticks(
    ticks=np.arange(len(cluster_cols)),
    labels=cluster_cols,
    rotation=45,
    ha="right"
)

plt.yticks(
    ticks=np.arange(len(plot_data.index)),
    labels=plot_data.index
)

plt.xlabel("Visual clusters")
plt.ylabel("Parks")
plt.title("Park-level Visual Cluster Composition (K=6)")

for i in range(len(plot_data.index)):
    for j in range(len(cluster_cols)):
        current_value = plot_data.iloc[i, j] # Use original plot_data for text conditions
        # Display text only if the value is greater than 0.0001
        if current_value > 0.0001:
            # Determine text color based on the actual value (for readability on colored background)
            # If the background is white (value < 0.01), text should be black.
            # If the background is colored (value >= 0.01), text color depends on value intensity.
            if current_value < 0.01: # On white background, text should be black
                text_color = 'black'
            else: # On colored background, choose based on intensity
                text_color = 'white' if current_value > 0.2 else 'black'

            plt.text(j, i, f'{current_value:.4f}', # Keeping .4f for four decimal places
                     ha="center", va="center", color=text_color, fontsize=8)

plt.tight_layout()

heatmap_output = output_dir / f"park_cluster_ratio_heatmap_k6_{timestamp}.png"
plt.savefig(heatmap_output, dpi=300, bbox_inches="tight")
plt.show()

print("热力图已保存:", heatmap_output)

# =========================
# 8. 堆叠柱状图 (Reuse existing code with new data)
# =========================

fig, ax = plt.subplots(figsize=(14, 7))

bottom = np.zeros(len(bar_data))

# Use a colormap for the bars
colors = plt.cm.get_cmap('viridis', len(cluster_cols))

for i, c in enumerate(cluster_cols):
    ax.bar(
        bar_data.index,
        bar_data[c],
        bottom=bottom,
        label=c,
        color=colors(i) # Assign color from colormap
    )
    bottom += bar_data[c].values

ax.set_ylabel("Cluster proportion")
ax.set_xlabel("Parks")
ax.set_title("Park-level Visual Cluster Composition (K=6)")
ax.set_ylim(0, 1)

plt.xticks(rotation=90)
ax.legend(
    title="Cluster",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()

bar_output = output_dir / f"park_cluster_ratio_stacked_bar_k6_{timestamp}.png"
plt.savefig(bar_output, dpi=300, bbox_inches="tight")
plt.show()

print("堆叠柱状图已保存:", bar_output)

### 9. Export Cluster Proportions to Excel

In [ ]:
from pathlib import Path
import datetime

# Re-define output_dir and timestamp for self-contained execution
output_dir = Path("/content/drive/MyDrive/CES分析/图片最终分析RAM/DINOv生成数据2/cluster可视化")
output_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

excel_output_path = output_dir / f"park_cluster_proportions_k6_{timestamp}.xlsx"
cluster_proportions.to_excel(excel_output_path, index=True)
print(f"每个公园各个cluster占比的Excel表格已保存: {excel_output_path}")